[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/superfluid/superfluid.ipynb)

# Two-Fluid Hydrodynamics of Superfluid Helium

This example computes a pulse of heat in superfluid helium, let off in a cavity divided by a wall of fine powder, and the fountain effect across that wall. Below about 2.17 kelvin, liquid helium flows as two fluids through each other: a superfluid, which carries mass but no entropy and flows without viscosity, and a normal fluid, which carries the rest of the mass and all of the entropy. Heat does not diffuse through it: a warm place pushes its normal fluid out and draws superfluid in, and this counterflow runs on as a wave of temperature, second sound, at about twenty metres per second. A fine powder holds the normal fluid still and lets the superfluid through, so it stops heat but not pressure: the superfluid is drawn through it towards a warm side and raises the pressure there, the fountain effect. [Donnelly (2009)](https://doi.org/10.1063/1.3248499) recounts how Tisza's two-fluid picture and Landau's theory came to predict second sound; the liquid's properties here are the measurements compiled by [Donnelly and Barenghi (1998)](https://doi.org/10.1063/1.556028).

The algebra is the plane joined with two degenerate directions, one for mass and one for entropy, `Algebra("x+y+") * Algebra("m0s0")`. A parcel of the liquid is a point weighed by the mass and the entropy it holds, `mass * mv.m.dual() + entropy * mv.s.dual()`, and a superfluid parcel's entropy is zero. The potentials, the chemical potential per mass along `m` and the temperature along `s`, are one vector, and their change read by the whole liquid's parcel, `potentials & liquid`, is the change of its pressure, the Gibbs–Duhem relation as one pairing. Their drive is minus their gradient wedged with them, and its commutator with a parcel, `drive.commutator(parcel)`, is the force on what the parcel holds: the two-fluid model's pair of forces is one product. On a mesh of triangles a parcel's flux is `velocity.regressive(parcel)`, and the continuity that fills and empties the cells reads that flux through the same gradients as the drive, so the liquid keeps the energy it is given.

In the notation of the two-fluid model, without the normal fluid's viscosity, the components accelerate as $\rho_s\,\partial_t\mathbf v_s = -\frac{\rho_s}{\rho}\nabla p + \rho_s s\nabla T$ and $\rho_n\,\partial_t\mathbf v_n = -\frac{\rho_n}{\rho}\nabla p - \rho_s s\nabla T$, which the Gibbs–Duhem relation $\nabla p = \rho\nabla\mu + \rho s\nabla T$ turns into $-\rho_s\nabla\mu$ and $-\rho_n\nabla\mu - \rho s\nabla T$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_animation
from examples.fluids.superfluid import render, scenarios

# The plane, and the mass and the entropy a parcel holds: each a direction of its own, degenerate, since nothing
# measures a mass or an entropy against a length.
space = Algebra("x+y+")
amounts = Algebra("m0s0")
ga = space * amounts
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# What a parcel holds, the points that span the plane, and the potentials that read it.
Amount = Plane.derive.within(amounts).derive.dual                             # Antivector-
Potential = Plane.derive.within(amounts)                                      # Vector+
Drive = ga.gatype.bivector().derive.touching(space, amounts)                  # Bivector+
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
# A unit of mass, the weight of a place, and a unit of entropy.
mass = mv.m.dual()                                                            # [] Amount-
entropy = mv.s.dual()                                                         # [] Amount-

## Two fluids

A drive pushes each component of the liquid by what its parcels hold. The superfluid's parcels hold mass alone, so the drive of a temperature passes over them and pushes only the normal fluid, through its entropy, while the drive of a chemical potential pushes both components by their mass. The whole liquid, read by its own parcel, is pushed by the fall of its pressure. The liquid's properties are tabulated at temperatures from 1.2 to 1.9 kelvin, in SI units; the components' parcels are per cubic metre.

In [ ]:
# The liquid over temperature: its density and its normal fluid's, its first sound, its heat capacity per mass and its
# entropy per mass; the cavity's temperature, 1.8 kelvin, as an index into them.
temperatures, densities, normal_densities = scenarios.temperatures, scenarios.densities, scenarios.normal_densities
first_sounds, heat_capacities, entropies = scenarios.first_sounds, scenarios.heat_capacities, scenarios.entropies
at = scenarios.at

# The superfluid's parcels hold its mass alone, the normal fluid's the rest of the mass and all of the entropy.
masses = np.stack([densities - normal_densities, normal_densities], axis=-1)  # [temperatures, components]
held_entropies = np.stack([0 * entropies, densities * entropies], axis=-1)    # [temperatures, components]
parcels = masses * mass + held_entropies * entropy                            # [temperatures, components] Amount-
# The drives of a chemical potential and of a temperature rising by one along x.
drives = stack((-mv.xm, -mv.xs))                                              # [potentials] Drive+
forces = drives[:, None].commutator(parcels[at][None, :])                     # [potentials, components] Point-

In [ ]:
np.set_printoptions(precision=1, suppress=True)
print("pushes along x, of the chemical potential's and the temperature's drive (rows) on the superfluid and the normal fluid (columns):")
print(mv.x & forces)

## The stiffness

The liquid's thermodynamic stiffness maps the mass and the entropy a volume holds to the change of its potentials. Compression raises the chemical potential; heating, a rise of the entropy per mass, raises the temperature and lowers the chemical potential by the entropy per mass times that rise. Each is a dyad, a potential times its own reading of the amounts, `heating * (heating & Amount)`, so the stiffness is symmetric, as Maxwell's relations require.

In [ ]:
# Heating: a unit rise of the entropy per mass, read as the change of entropy beyond what the mass carries in.
heating = mv.s - entropies * mv.m                                             # [temperatures] Potential+
stiffnesses = (first_sounds ** 2 / densities) * (mv.m * (mv.m & Amount)) + (
    temperatures / (heat_capacities * densities)) * (heating * (heating & Amount))   # [temperatures] Potential+ <- Amount-

## The cavity

The cavity is a rectangle of triangles, with a wall of powder across it and a heater in the larger chamber. The mass and the entropy per volume live at the vertices, each vertex a place of unit mass, and each component has one velocity in each triangle. A corner's hat gradient is the plane through the facing side, at unit entropy and mass, over twice the triangle's area, so that it reads one at the corner. Lengths are in millimetres and times in milliseconds, so that speeds stay in metres per second.

In [ ]:
# The cavity's grid of triangles: its topology, its vertices' places, which triangles lie in the powder, and the
# heater's pulse of temperature at each vertex.
topology, places, in_powder, rise = scenarios.scene()
vertices = (mass + axis * places[:, 0] + across * places[:, 1]).field()       # Point-[vertices]
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)[..., None]      # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, read at unit entropy and mass.
double_areas = ((sides[..., 0] & sides[..., 1]) & entropy) & mass             # Scalar-[faces]
areas = double_areas.norm() / 2                                               # Scalar+[faces]
gradients = (entropy & mass & sides) / double_areas[..., None]                # [corners] Plane+[faces]
# Each vertex's share of the area around it.
cells = (~corners * (areas / 3)).sum(axis=-1)                                 # Scalar+[vertices]
render.draw_cavity(topology, vertices, in_powder);

The cavity's triangles, with those in the powder pale tan.

## A heat pulse

The liquid moves by its drive and its continuity alone. The normal fluid's viscosity, its conduction of heat and the mutual friction between the two fluids are left out, so nothing damps the motion: the waves ring on, and the pressure sloshes between the chambers rather than settling. The drive pushes each component by what its parcels hold, and the flux the parcels carry empties the cells it runs out of into those it runs into. A hat gradient wedged with a flux, `gradient ^ flux`, is what the flux carries out of a corner's cell; paired with the potentials it is the drive's own reading of the same flux, so the energy the cells lose is the energy the components gain. In the powder the normal fluid is held still. The heater adds entropy at constant density, a millikelvin's rise at its middle, and the liquid's temperature and pressure are read from its potentials: the temperature by a unit of entropy, the pressure by the whole liquid's parcel. This cell takes about ten seconds, most of it stepping the liquid and drawing the animation.

In [ ]:
# The liquid at 1.8 kelvin; the time step in milliseconds, half the time first sound takes across a triangle's side;
# the steps, and the steps between frames.
stiffness = stiffnesses[at]                                                   # [] Potential+ <- Amount-
liquid_parcels = parcels[at]                                                  # [components] Amount-
dt = scenarios.courant * (scenarios.width / (scenarios.n_columns - 1)) / first_sounds[at]
n_steps, stride = scenarios.n_steps, scenarios.stride
# How freely each component moves in each triangle: the superfluid everywhere, the normal fluid but in the powder.
held = mv.scalar(in_powder[:, None] * 1.0).field()                            # Scalar+[faces]
freedom = (1 - held[..., None] * np.array([0.0, 1.0])) / masses[at]           # [components] Scalar+[faces]


def stepped(amounts: Point, supply: Point) -> Iterator[Point]:
    """The mass and the entropy per volume over the cavity `Point-[vertices]`, from these at rest, every `stride`
    steps, with this supply of them per volume and time `Point-[vertices]`."""
    velocities = 0 * (axis * freedom)                                         # [components] Point-[faces]
    for step in range(n_steps + 1):
        if step % stride == 0:
            yield amounts
        potentials = stiffness(amounts)                                       # Potential+[vertices]
        drive = -(gradients ^ (corners * potentials[..., None])).sum(axis=-1)   # Drive+[faces]
        velocities = velocities + dt * freedom * drive.commutator(liquid_parcels)   # [components] Point-[faces]
        flux = velocities.regressive(liquid_parcels).sum(axis=-1)            # Flux-[faces]
        # What the flux carries out of each corner's cell, and what the supply adds to it.
        amounts = amounts + dt * (supply - (~corners * (areas * (gradients ^ flux[..., None]))).sum(axis=-1) / cells)


# The entropy that raises the temperature by the pulse at constant density.
added = rise * densities[at] * heat_capacities[at] / temperatures[at]       # [vertices]
states = list(stepped((0 * mass + entropy * added).field(), (0 * mass + 0 * entropy * added).field()))   # [frames] Point-[vertices]
liquid = liquid_parcels.sum(axis=-1)                                          # [] Amount-
readings = [(stiffness(amounts) & entropy, stiffness(amounts) & liquid) for amounts in states]
# The rises at which the colours saturate: shares of the pulse's rise of temperature and of its fountain pressure, the
# density times the entropy per mass times that rise.
warmest = scenarios.pulse_temperature_share * scenarios.pulse_rise
highest = scenarios.pulse_pressure_share * densities[at] * entropies[at] * scenarios.pulse_rise
show_animation(render.animate(topology, vertices, in_powder, readings, warmest, highest), "superfluid_pulse",
               scenarios.frame_duration_ms)

The top picture is the rise of temperature, red where the liquid is warmer and blue where cooler, and the bottom one the rise of pressure, orange where it is up and purple where down, with the powder between the grey strokes. The pulse leaves as a ring of second sound with a cool wake behind it, and the powder reflects it like a wall. The pressure stays flat until the ring reaches the powder; then the superfluid drawn towards the warm face runs through the powder, and the pressure sloshes between the two chambers. The faint ripples of temperature in the far chamber are second sound too, raised where the superfluid streams out of the powder into liquid whose normal fluid is at rest, as a superleak's transducer raises it; no entropy crosses the powder, so on balance they carry no heat.

## The fountain effect

Left on, the heater warms its chamber steadily, and no entropy leaves it through the powder. The superfluid in the powder is pushed by the fall of the chemical potential alone, and the chemical potential is level across the powder when the pressure on the warm side exceeds that on the cold side by the density times the entropy per mass times the difference of temperature, London's fountain pressure. Read by the whole liquid's parcel, a level chemical potential with a step of temperature is that step of pressure, so the fountain pressure is the Gibbs–Duhem pairing with the chemical potential held level. The chambers' difference of pressure follows it once the heat has reached the powder, the superfluid sloshing through the powder about it. The heater's rate warms its chamber by a millikelvin per millisecond on average, spread like the pulse. This cell takes about fifteen seconds, most of it stepping the liquid and drawing the animation.

In [ ]:
# The heater's supply of entropy per volume and time: the pulse's shape, scaled so that it warms its chamber at the
# warming rate on average, in kelvin per millisecond.
spacing = (scenarios.width / (scenarios.n_columns - 1)) * (scenarios.height / (scenarios.n_rows - 1))
chamber_area = (scenarios.wall_middle - scenarios.wall_half_thickness) * scenarios.height
warming = rise / (rise.sum() * spacing) * scenarios.warming_rate * chamber_area   # [vertices]
supplied = warming * densities[at] * heat_capacities[at] / temperatures[at]  # [vertices]
heated_states = list(stepped((0 * mass + 0 * entropy * supplied).field(), (0 * mass + entropy * supplied).field()))
heated_readings = [(stiffness(amounts) & entropy, stiffness(amounts) & liquid) for amounts in heated_states]
times = np.arange(len(heated_states)) * stride * dt                           # [frames]
# The colours saturate at a multiple of the chamber's last mean rise of temperature and of its fountain pressure.
heated_warmest = scenarios.heater_share * scenarios.warming_rate * times[-1]
show_animation(render.animate(topology, vertices, in_powder, heated_readings, heated_warmest,
                              densities[at] * entropies[at] * heated_warmest), "superfluid_heater",
               scenarios.frame_duration_ms)

The two pictures read as before. The heat fills its chamber as second sound, ringing back and forth rather than spreading, and stays there; the pressure rises on the warm side and falls on the cold one.

In [ ]:
# checks
# The temperature's drive passes over the superfluid and pushes the normal fluid by its entropy; the chemical
# potential's pushes each component by its mass.
np.testing.assert_allclose((mv.x & forces).to_array(), -np.stack([masses[at], held_entropies[at]]), rtol=1e-12)
# The whole liquid is pushed by the fall of its pressure, the potentials read by its parcel: the Gibbs-Duhem relation.
np.testing.assert_allclose((mv.x & drives.commutator(liquid)).to_array(),
                           -(stack((mv.m, mv.s)) & liquid).to_array(), rtol=1e-12)
# The cavity keeps its mass and its entropy.
totals = stack([(cells * amounts).sites.sum() for amounts in (states[0], states[-1])])   # [2] Amount-
np.testing.assert_allclose((stack((mv.m, mv.s)) & totals[:, None]).to_array()[1],
                           (stack((mv.m, mv.s)) & totals[:, None]).to_array()[0], rtol=1e-10, atol=1e-10)
# No entropy crosses the powder: the far chamber, from the powder's far face on, holds none at any frame.
far = mv.scalar((places[:, 0] >= scenarios.wall_middle + scenarios.wall_half_thickness)[:, None] * 1.0).field()
far_entropies = stack([(far * cells * (mv.s & amounts)).sites.sum() for amounts in states])   # [frames] Scalar+
added_entropy = (cells * (mv.s & states[0])).sites.sum()                                       # [] Scalar+
assert np.abs(far_entropies.to_array()).max() < 1e-12 * added_entropy.to_array()
# The difference between a field's mean over the heated chamber and over the far one, each vertex weighed by its share
# of the area.
hot = mv.scalar((places[:, 0] < scenarios.wall_middle - scenarios.wall_half_thickness)[:, None] * 1.0).field()
cold = mv.scalar((places[:, 0] > scenarios.wall_middle + scenarios.wall_half_thickness)[:, None] * 1.0).field()
chambers = cells * hot / (cells * hot).sites.sum() - cells * cold / (cells * cold).sites.sum()   # Scalar+[vertices]
pressure_differences = stack([(chambers * pressure).sites.sum() for _, pressure in heated_readings])   # [frames] Scalar+
# The fountain pressure: the density times the entropy per mass times the chambers' difference of temperature.
fountain_pressures = densities[at] * entropies[at] * stack(
    [(chambers * warmth).sites.sum() for warmth, _ in heated_readings])      # [frames] Scalar+
# Over the heater's second half, the chambers' difference of pressure is the fountain pressure on average, within the
# sloshing's share of a few periods.
later = len(times) // 2
np.testing.assert_allclose(pressure_differences.to_array()[later:].mean(), fountain_pressures.to_array()[later:].mean(),
                           rtol=0.15)